# What would that order have cost? The public sample day, no key

This notebook uses only the free Polymarket sample day (Bitcoin Up or Down, 4-hour markets, 19 August 2026) and the TickFoundry package. It needs no account and no API key.

It loads the day, picks the busiest outcome token, and asks the Lab's own simulator what a $100, $1,000 and $10,000 market buy would have paid at one moment, what it would have cost to move the price by 2 cents, and what a round trip and a sliced order cost.

In [ ]:
import tickfoundry as tf

d = tf.sample()          # downloads and caches the public sample zip
d

In [ ]:
# One LocalReplay per bundle day: the Lab's own replay and simulation code.
r = tf.LocalReplay.from_bundle(d.folders[0])
busiest = max(r.instrument_ids(), key=lambda i: r.books(i).n)
t = "2026-08-19T21:00:00Z"
busiest, r.describe(busiest)

In [ ]:
rows = []
for budget in ("100", "1000", "10000"):
    s = r.simulate(busiest, t, side="buy", size_mode="quote_budget", size=budget)
    res = s.result
    rows.append({"budget_usd": budget, "status": s.status, "contracts": res.filled_quantity, "avg_price": res.vwap,
                 "slippage_bps": res.relative_slippage_bps, "levels_used": res.levels_consumed})
import pandas as pd
pd.DataFrame(rows)

In [ ]:
# Dollars needed to push the price 2 cents up (and down), on the book in force at t.
m = r.cost_to_move(busiest, t, cents="0.02")
m.status, m.result.mid, m.result.up, m.result.down

In [ ]:
# Buy 500 contracts and sell them a minute later; then work 2,000 contracts in 5 slices over 10 minutes.
rt = tf.round_trip(r, busiest, t, size="500", hold_ms=60_000)
so = tf.sliced_order(r, busiest, t, "2026-08-19T21:10:00Z", side="buy", total_size="2000", slices=5)
rt.to_dict(), (so.vwap, so.shortfall_bps, so.single_shot_shortfall_bps, so.note)

## Read the limits before using a number

Every number walks the order book as it was recorded. Our order never touched that book, so nothing here models our own market impact, the book refilling after us, queue position or hidden liquidity. A sliced order's cost is therefore a lower bound. The code is MIT-licensed; the data is covered by the TickFoundry data licence of your plan (the sample day by the free sample's terms).

With an API key, `tf.pull(series_id, start, end)` loads any range your plan covers in the same shape.